# 01 — Sanity Check: tokenizer wrappers on Korean

**Goal**: confirm each provider's `Tokenizer` wrapper produces token counts in the expected range for Korean text *before* we trust any later experiment. New wrappers are added one at a time; PASS here is the gate to add the next.

**Per-provider expected TPC range** (calibrated on n=10, 2026-05-07). Each tokenizer has its own band — Claude's Korean tokenizer is expected to fall back to byte-level on Hangul, so a TPC > 1.0 is *normal Claude behavior*, not a wrapper bug.

| Provider | Korean | English |
|---|---|---|
| GPT-4o (`o200k_base`) | 0.5 – 0.8 | 0.15 – 0.30 |
| Claude Sonnet 4.5 | 0.9 – 1.3 | 0.20 – 0.35 |
| Google / HF | TBD on first run | TBD on first run |

**Phase 1 surprise hypotheses** (we are not testing them here yet, but later runs will):

- **H1** — frontier models meaningfully improved Korean efficiency  
- **H2** — Korean-specialized models win on TPC but lose on ECPC after pricing  
- **H3** — medical / 한자-mixed text has a far higher penalty than everyday text

**Inter-model signal threshold (project decision rule)**: a Claude/GPT-4o Korean-TPC ratio of **≥ 1.3×** at full-corpus scale would count as a real H1/H2 signal. Below 1.3× we pivot to domain-variance (H3) as the paper's primary angle. At n=10 the ratio is informational only — never paper-cite.

**Asymmetry test (added 2026-05-07 after observing Claude Korean TPC = 1.074)**: compare Claude/GPT ratio on Korean vs English. If KO ratio ≫ EN ratio (≥ 1.3× asymmetry), the paper headline becomes "Korean-*specific* tokenizer gap" — sharper hook than "general inefficiency".

In [9]:
from __future__ import annotations

import sys
from pathlib import Path

# Make the in-tree package importable without installing it.
# Once you `uv sync` and `pip install -e .` you can drop these two lines.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from korean_llm_cost.tokenizers.openai_tok import OpenAITokenizer

tok = OpenAITokenizer("gpt-4o")
print(tok)

<OpenAITokenizer gpt-4o@2024-11-20>


In [10]:
# 10 Korean samples across categories we plan to measure in Phase 1,
# plus a few harder cases (한자 mixed, code mixed, exclamation) to see
# the per-sample variance even at this tiny scale.
korean_samples: list[tuple[str, str]] = [
    ("conv",       "오늘 점심 뭐 먹을까?"),
    ("conv",       "내일 회의 시간 좀 알려줘."),
    ("news",       "한국은행은 기준금리를 동결하기로 결정했다."),
    ("news",       "정부는 내년도 예산안을 국회에 제출하였다."),
    ("medical",    "환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다."),
    ("medical",    "고혈압성 좌심실 비대 소견이 관찰되었다."),
    ("academic",   "본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다."),
    ("code_mixed", "리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다."),
    ("hanja",      "大韓民國의 헌법 제1조는 민주공화국임을 천명한다."),
    ("short",      "와 진짜 대박이네요!"),
]

english_baseline: list[str] = [
    "What should we eat for lunch today?",
    "Let me know the meeting time tomorrow.",
    "The Bank of Korea decided to freeze the benchmark interest rate.",
    "This study quantitatively analyzes the efficiency of Korean tokenizers.",
    "Wow, that's really amazing!",
]

len(korean_samples), len(english_baseline)

(10, 5)

In [11]:
def tpc(text: str) -> float:
    """Tokens per character. Aggregate over many texts via summed counts,
    not averaged TPCs — averaging per-text TPC over-weights short texts."""
    return tok.count(text) / len(text) if text else 0.0


print(f"{'category':<12} {'chars':>5} {'toks':>5} {'tpc':>6}  text")
print("-" * 80)
for category, text in korean_samples:
    n_chars = len(text)
    n_toks = tok.count(text)
    print(f"{category:<12} {n_chars:>5} {n_toks:>5} {n_toks/n_chars:>6.3f}  {text}")

category     chars  toks    tpc  text
--------------------------------------------------------------------------------
conv            12     8  0.667  오늘 점심 뭐 먹을까?
conv            15     9  0.600  내일 회의 시간 좀 알려줘.
news            23    13  0.565  한국은행은 기준금리를 동결하기로 결정했다.
news            23    13  0.565  정부는 내년도 예산안을 국회에 제출하였다.
medical         40    30  0.750  환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다.
medical         22    17  0.773  고혈압성 좌심실 비대 소견이 관찰되었다.
academic        33    20  0.606  본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다.
code_mixed      38    22  0.579  리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다.
hanja           27    22  0.815  大韓民國의 헌법 제1조는 민주공화국임을 천명한다.
short           11     8  0.727  와 진짜 대박이네요!


In [ ]:
# Aggregate TPC: total tokens / total chars (NOT mean of per-text TPCs).
ko_chars = sum(len(t) for _, t in korean_samples)
ko_toks = sum(tok.count(t) for _, t in korean_samples)
en_chars = sum(len(t) for t in english_baseline)
en_toks = sum(tok.count(t) for t in english_baseline)

ko_tpc = ko_toks / ko_chars
en_tpc = en_toks / en_chars
kpr_gpt = ko_tpc / en_tpc  # rough; real KPR needs aligned parallel data

print(f"GPT-4o Korean : {ko_toks:>4} tokens / {ko_chars:>4} chars = TPC {ko_tpc:.3f}")
print(f"GPT-4o English: {en_toks:>4} tokens / {en_chars:>4} chars = TPC {en_tpc:.3f}")
print(f"GPT-4o KPR (ko / en) = {kpr_gpt:.2f}×")

# Pass / fail against GPT-4o-specific ranges (each provider gets its own band).
ok_ko = 0.5 <= ko_tpc <= 0.8
ok_en = 0.15 <= en_tpc <= 0.30
print()
print(f"GPT-4o Korean  TPC in [0.50, 0.80]? ", "PASS" if ok_ko else f"FAIL ({ko_tpc:.3f})")
print(f"GPT-4o English TPC in [0.15, 0.30]? ", "PASS" if ok_en else f"FAIL ({en_tpc:.3f})")

## Anthropic (Claude) check — including English baseline

Same 10 Korean + 5 English samples through the Claude `count_tokens` API. The added English measurement lets us run the *asymmetry test*: compare Claude/GPT ratios on Korean vs English to tell apart "Korean-specific gap" from "general Claude inefficiency".

**Prereq**: `uv sync --extra anthropic --extra dev`, `ANTHROPIC_API_KEY` in env, **and a non-zero account balance** at console.anthropic.com.

**Cost**: `count_tokens` has zero per-call billing, but Anthropic's API requires a positive balance to access at all (otherwise you get HTTP 400 `"credit balance is too low"`). A one-time minimum top-up (e.g. $5) covers this entire project — Phase 1 token-counting will not draw it down meaningfully.

In [ ]:
import os

anth = None
try:
    from korean_llm_cost.tokenizers.anthropic_tok import AnthropicTokenizer
except ImportError:
    print("anthropic SDK not installed. Run:  uv sync --extra anthropic --extra dev")
else:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("ANTHROPIC_API_KEY not set.")
        print("  cp .env.example .env  →  fill in the key  →  restart kernel.")
    else:
        # Default to Sonnet 4.5 (stable). Override to claude-sonnet-4-7 once verified.
        anth = AnthropicTokenizer("claude-sonnet-4-5")
        print(anth, f"(chat overhead = {anth._overhead} tokens, calibrated)")

In [ ]:
if anth is not None:
    # ~15 API calls total. Zero per-call billing.
    cla_ko_counts = [anth.count(text) for _, text in korean_samples]
    cla_en_counts = [anth.count(text)  for text in english_baseline]

    # ----- Korean per-sample table -----
    print("=== Korean (10 samples) ===")
    print(f"{'category':<12} {'chars':>5} {'gpt':>4} {'cla':>4}  text")
    print("-" * 80)
    for (category, text), n_cla in zip(korean_samples, cla_ko_counts):
        n_chars = len(text)
        n_gpt = tok.count(text)
        print(f"{category:<12} {n_chars:>5} {n_gpt:>4} {n_cla:>4}  {text}")

    # ----- English per-sample table -----
    print("\n=== English (5 samples) ===")
    print(f"{'chars':>5} {'gpt':>4} {'cla':>4}  text")
    print("-" * 80)
    for text, n_cla in zip(english_baseline, cla_en_counts):
        n_chars = len(text)
        n_gpt = tok.count(text)
        print(f"{n_chars:>5} {n_gpt:>4} {n_cla:>4}  {text}")

    # ----- Aggregate TPCs (4 numbers) -----
    gpt_ko_tpc = ko_tpc                      # from earlier cell
    gpt_en_tpc = en_tpc                      # from earlier cell
    cla_ko_tpc = sum(cla_ko_counts) / ko_chars
    cla_en_tpc = sum(cla_en_counts) / en_chars

    print("\n=== Aggregate TPC ===")
    print(f"  GPT-4o   Korean : {gpt_ko_tpc:.3f}")
    print(f"  GPT-4o   English: {gpt_en_tpc:.3f}")
    print(f"  Claude   Korean : {cla_ko_tpc:.3f}")
    print(f"  Claude   English: {cla_en_tpc:.3f}")

    # ----- Cross-model ratios (Claude / GPT-4o) -----
    ratio_ko = cla_ko_tpc / gpt_ko_tpc
    ratio_en = cla_en_tpc / gpt_en_tpc
    asymmetry = ratio_ko / ratio_en  # > 1 → Claude relatively worse on Korean

    print("\n=== Inter-model ratios (Claude / GPT-4o) ===")
    print(f"  Korean  ratio: {ratio_ko:.2f}×")
    print(f"  English ratio: {ratio_en:.2f}×")
    print(f"  Asymmetry (KO ratio / EN ratio): {asymmetry:.2f}×")

    # ----- One-line asymmetry verdict -----
    if asymmetry >= 1.3:
        verdict = "Korean-SPECIFIC gap — Claude penalty on Korean far exceeds its English penalty."
        impl = "Strong paper hook: tokenizer disparity is *language-specific*, not general."
    elif asymmetry <= 1.0 / 1.3:
        verdict = "Inverted asymmetry — Claude is RELATIVELY worse on English than Korean."
        impl = "Investigate before reporting; sample bias likely. Re-check on a balanced corpus."
    else:
        verdict = "General Claude inefficiency — Claude is uniformly costlier across both languages."
        impl = "Paper hook softer: 'Claude general tokenizer gap', not 'Korean-specific'."
    print(f"\n→ {verdict}")
    print(f"  {impl}")

    # ----- Per-provider sanity (each model has its own expected band) -----
    print("\n=== Sanity (per-provider expected ranges) ===")
    checks = [
        ("GPT-4o Korean ", gpt_ko_tpc, 0.50, 0.80),
        ("GPT-4o English", gpt_en_tpc, 0.15, 0.30),
        ("Claude Korean ", cla_ko_tpc, 0.90, 1.30),
        ("Claude English", cla_en_tpc, 0.20, 0.35),
    ]
    for label, val, lo, hi in checks:
        ok = lo <= val <= hi
        status = "PASS" if ok else f"FAIL ({val:.3f})"
        print(f"  {label} TPC in [{lo:.2f}, {hi:.2f}]?  {status}")

    # ----- Project signal threshold (1.3× on Korean ratio) -----
    print()
    if abs(ratio_ko - 1.0) >= 0.3:
        print(f"→ Korean ratio |{ratio_ko:.2f} − 1| ≥ 0.3 at n=10. Candidate H1/H2 signal.")
    else:
        print(f"→ Korean ratio |{ratio_ko:.2f} − 1| < 0.3 at n=10. No clear inter-model gap.")

## What to do if a check fails

Each provider has its own TPC band — a Claude Korean TPC of 1.07 is **expected behavior**, not a wrapper bug. Only a value far outside the provider's band is a real failure.

- **Korean TPC outside the provider's band by > 30%**: usually encoding (UTF-8 vs NFD), wrong model name, or `count` being called on bytes. Print `tok.info` and confirm the snapshot.
- **Claude overhead negative** (overhead = -1, -2…): the 1-character probe assumption (`'.' = 1 token`) may have broken in a future Claude release. Re-derive overhead by sending two probes of known different lengths and solving for the constant.
- **Claude HTTP 400 "credit balance is too low"**: API key works but balance is empty. Top up at console.anthropic.com. `count_tokens` itself is free.
- **English TPC < 0.15**: extremely common-word content; the 5-sentence sanity sample is unrepresentatively easy. Not blocking — verify on the news/academic corpus before reporting.

## Next

Once GPT-4o + Claude both PASS their bands, the next wrapper to add is **Google Gemini** (same pattern — own cells, own PASS gate). After Google: HuggingFace wrappers one model at a time (Solar / Llama / Qwen). Only when all six Phase 1 wrappers have PASS rows in this notebook do we move on to `metrics.py` and corpus collection.

## Early signals (n=10 / n=5, **NOT** for paper)

The sanity sample is too small for any number to appear in the paper. The signals below are **candidates** to verify on the full Phase 1 corpus (≥ 1000 sentences/category, bootstrap CI, paired tests). They live here so we don't lose track of where to look first.

### Within-model (GPT-4o, n=10)

| Candidate signal | Observed | Verify on |
|---|---|---|
| Korean ≈ 3.4× English per char (KPR) | 3.39 | parallel ko↔en corpus (KorNLI/MNLI overlap, OpenSubtitles ko-en) |
| Hanja-mixed text exceeds 0.8 TPC | 0.815 | hanja-rich subcorpus (legal / historical text) |
| Medical 13–17% above conversational | 0.75–0.77 vs 0.60–0.67 | KorMedMCQA + AI Hub medical text |
| Code-mixed *lower* than expected | 0.579 (English identifiers like `useState` likely 1 token) | dev-blog crawl (Velog/Tistory) |

### Cross-model (GPT-4o vs Claude Sonnet 4.5)

| Candidate signal | Observed | Verify on |
|---|---|---|
| Claude Korean TPC ≈ 62% above GPT-4o | 1.074 vs 0.664 (ratio **1.62×**, n=10) | full Phase 1 corpus, 6 models × 3 categories |
| Claude / GPT-4o **English** ratio | _filled in by Anthropic cell on this run_ | parallel ko↔en corpus |
| Asymmetry (KO ratio / EN ratio) | _filled in by Anthropic cell on this run_ | full Phase 1 corpus |

**Interpretation rules** (so we read the next-run output the same way each time):
- **Asymmetry ≥ 1.3×** → "Korean-specific gap", paper hook is sharp
- **Asymmetry between 1.0× and 1.3×** → "Claude general inefficiency", softer hook
- **Asymmetry ≤ 1/1.3×** → unexpected, investigate sample bias

**Project signal threshold (decision rule, 2026-05-07):**
- Inter-model TPC ratio **≥ 1.3×** at full-corpus scale → H1/H2 candidate signal, primary paper angle
- Inter-model TPC ratio **< 1.3×** → pivot toward domain-variance (H3) as the primary angle

**Do not cite n=10 numbers anywhere outside this notebook.** They exist only to motivate which directions are worth prioritizing in the full Phase 1 run.